In [3]:
"""
Low-Rank Factorization (SVD / Tensor Decomposition) — ResNet Model Compression / CIFAR-10
===========================================================================================

WHAT THIS SCRIPT DOES
=====================
Implements three low-rank decomposition variants applied to a trained ResNet model,
sweeping over all three:

  1. SVD-CONV   — Truncated SVD on conv weight tensors reshaped to 2D matrices.
                  Each (C_out, C_in·kH·kW) matrix is factored as U·Σ·Vᵀ, keeping
                  top-r singular values. Replaced by two consecutive conv layers:
                    - Conv2d(C_in, r, 1×1)   [captures V side]
                    - Conv2d(r, C_out, kH×kW) [captures U·Σ side]

  2. CP-APPROX  — CANDECOMP/PARAFAC decomposition on conv weight tensors.
                  Approximates W ∈ R^(C_out × C_in × kH × kW) as sum of r rank-1 terms:
                    W ≈ Σᵢ aᵢ⊗bᵢ⊗cᵢ⊗dᵢ
                  Implemented via sequential 1×1 → kH×1 → 1×kW → 1×1 convolutions.

  3. TUCKER-2   — Tucker decomposition on the (C_out, C_in) modes only.
                  W ∈ R^(C_out × C_in × kH × kW) ≈ G ×₁ A ×₂ B
                  where G ∈ R^(r_out × r_in × kH × kW), A ∈ R^(C_out × r_out), B ∈ R^(C_in × r_in).
                  Replaced by: Conv2d(C_in, r_in, 1×1) → Conv2d(r_in, r_out, kH×kW) → Conv2d(r_out, C_out, 1×1)

ARCHITECTURE
============
  Original model : ResNet (ResNet50 or ResNet18) trained on CIFAR-10
                   Loaded from ORIGINAL_MODEL_PATH below
                   Modified CIFAR-10 head: conv1→3×3 stride-1, maxpool→Identity, fc→10

  Compressed     : Same architecture with Conv2d layers replaced by low-rank equivalents.
                   No retraining by default; optional fine-tuning loop included.

OUTPUT
======
  __1__low_rank_decomposition.json  — mirrors KD JSON structure for direct comparison

MATH SUMMARY
============
  SVD: M = U·Σ·Vᵀ  →  keep top-r: M ≈ Uᵣ·Σᵣ·Vᵣᵀ
       Error: ‖M − Mᵣ‖²_F = σ²ᵣ₊₁ + … + σ²ₙ   (Eckart–Young theorem)

  Tucker: W ≈ G ×₁ A ×₂ B   (compress only channel modes, keep spatial intact)

  CP:     W ≈ Σᵢ₌₁ʳ λᵢ · aᵢ ⊗ bᵢ ⊗ cᵢ ⊗ dᵢ   (rank-1 outer products)
          Solved via ALS (Alternating Least Squares) iterations.

NOTE ON MODEL PATH
==================
  Set ORIGINAL_MODEL_PATH to point to your trained .pth file.
  If you used the same CIFAR-10 baseline setup, MODEL_ARCH should be "resnet50" or "resnet18".
"""

import os, json, time, copy, tempfile, warnings, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  : {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU      : {torch.cuda.get_device_name(0)}", flush=True)
    cap = torch.cuda.get_device_capability()
    print(f"[init] Compute  : SM{cap[0]}{cap[1]}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
# ↓↓ EDIT THESE to point to your files ↓↓
ORIGINAL_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"   # trained checkpoint
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"            # for accuracy_drop
MODEL_ARCH            = "resnet50"                                 # "resnet50" or "resnet18"
OUTPUT_JSON           = "__1__low_rank_decomposition.json"

NUM_CLASSES  = 10
BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"

# Rank ratio: keep this fraction of singular values / rank per layer.
# Lower → more compression, more accuracy drop.
# Well-tested range: 0.25 (aggressive) – 0.5 (conservative)
RANK_RATIO   = 0.25

# Optional fine-tuning after decomposition (set FINETUNE_EPOCHS=0 to skip)
FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-3

# ALS iterations for CP decomposition
CP_ALS_ITERS = 20

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] Model    : {MODEL_ARCH}  from {ORIGINAL_MODEL_PATH}", flush=True)
print(f"[init] RankRatio: {RANK_RATIO}  FineTuneEpochs: {FINETUNE_EPOCHS}", flush=True)


# ── Model builders ────────────────────────────────────────────────────────────
def build_model(arch: str, num_classes: int = 10) -> nn.Module:
    """Build CIFAR-10 modified ResNet (same as baseline script)."""
    if arch == "resnet50":
        m = models.resnet50(weights=None)
    elif arch == "resnet18":
        m = models.resnet18(weights=None)
    else:
        raise ValueError(f"Unknown arch: {arch}")
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model(path: str, arch: str) -> nn.Module:
    print(f"[model] Loading {arch} from {path} ...", flush=True)
    m = build_model(arch, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    m = m.to(DEVICE)
    m.eval()
    print(f"[model] Loaded. Params: {sum(p.numel() for p in m.parameters()):,}", flush=True)
    return m


# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_ld  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print(f"[data] Train batches: {len(train_ld)}  Test batches: {len(test_ld)}", flush=True)
    return train_ld, test_ld


# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)


def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device) -> dict:
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def measure_inference_ms(model: nn.Module, device, batch_size=128, runs=20) -> float:
    m = model.eval().to(device)
    dummy = torch.randn(batch_size, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(3): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000


def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()


def should_decompose(layer: nn.Conv2d) -> bool:
    """
    Skip layers where decomposition would be larger than original.
    1×1 convs are already rank-1 and gain nothing; very small channel counts
    would produce a rank below 1.
    """
    W = layer.weight                              # (C_out, C_in, kH, kW)
    C_out, C_in, kH, kW = W.shape
    # Skip 1×1 convs — no spatial structure to decompose
    if kH == 1 and kW == 1:
        return False
    # Skip if input channels is 3 (first conv, very thin)
    if C_in <= 3:
        return False
    # Compute rank for this layer
    r = max(1, int(min(C_out, C_in) * RANK_RATIO))
    # Check that decomposition would actually be smaller
    original_params   = C_out * C_in * kH * kW
    svd_params        = C_in * r + r * C_out * kH * kW   # SVD factored
    return svd_params < original_params


# ═══════════════════════════════════════════════════════════════════════════════
# DECOMPOSITION 1: SVD on Conv layers
# ═══════════════════════════════════════════════════════════════════════════════
class SVDConvReplacement(nn.Module):
    """
    Replaces a single Conv2d(C_in, C_out, kH, kW) with two convolutions
    using truncated SVD of the weight matrix.

    Reshape W: (C_out, C_in*kH*kW) → SVD → U(C_out, r), Σ(r,), Vᵀ(r, C_in*kH*kW)

    Factored as:
      1) Conv2d(C_in, r, kH×kW, ...)   ← absorbs Vᵀ  (spatial filtering)
      2) Conv2d(r, C_out, 1×1)         ← absorbs U·Σ  (channel mixing)

    This ordering keeps spatial convolution in the first (cheaper) layer.
    """
    def __init__(self, original_conv: nn.Conv2d, rank: int):
        super().__init__()
        W    = original_conv.weight.data              # (C_out, C_in, kH, kW)
        bias = original_conv.bias
        C_out, C_in, kH, kW = W.shape
        stride  = original_conv.stride
        padding = original_conv.padding
        groups  = original_conv.groups

        # Reshape to 2D matrix for SVD: (C_out, C_in·kH·kW)
        W_2d = W.reshape(C_out, -1).float()          # float32 for numerical stability

        # Truncated SVD — only compute top-r singular values/vectors
        # torch.linalg.svd gives full matrices; we slice to rank r
        U, S, Vh = torch.linalg.svd(W_2d, full_matrices=False)
        # U:  (C_out, min(C_out, C_in·kH·kW))
        # S:  (min(...),)
        # Vh: (min(...), C_in·kH·kW)
        r = min(rank, S.shape[0])

        U_r  = U[:, :r]             # (C_out, r)
        S_r  = S[:r]                # (r,)
        Vh_r = Vh[:r, :]            # (r, C_in·kH·kW)

        # Absorb singular values into U side: U_r · diag(S_r)  →  (C_out, r)
        US_r = U_r * S_r.unsqueeze(0)   # broadcast: (C_out, r) * (1, r)

        # First conv: captures Vᵀ — spatial filtering
        # Weight shape needed: (r, C_in, kH, kW)
        W_first = Vh_r.reshape(r, C_in, kH, kW)

        # Second conv: captures U·Σ — pointwise channel mixing
        # Weight shape needed: (C_out, r, 1, 1)
        W_second = US_r.reshape(C_out, r, 1, 1)

        self.conv1 = nn.Conv2d(C_in, r, kernel_size=(kH, kW),
                               stride=stride, padding=padding,
                               groups=groups, bias=False)
        self.conv2 = nn.Conv2d(r, C_out, kernel_size=1,
                               stride=1, padding=0, bias=(bias is not None))

        self.conv1.weight = nn.Parameter(W_first.to(original_conv.weight.dtype))
        self.conv2.weight = nn.Parameter(W_second.to(original_conv.weight.dtype))
        if bias is not None:
            self.conv2.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        return self.conv2(self.conv1(x))


def apply_svd_decomposition(model: nn.Module) -> nn.Module:
    """
    Walk all Conv2d layers and replace eligible ones with SVDConvReplacement.
    Uses recursive module replacement to handle nested modules (ResNet blocks).
    """
    model = copy.deepcopy(model)

    def _replace(parent: nn.Module, name: str, child: nn.Module):
        """Recursively replace eligible Conv2d modules."""
        for attr_name, submodule in child.named_children():
            _replace(child, attr_name, submodule)
        if isinstance(child, nn.Conv2d) and should_decompose(child):
            C_out, C_in, kH, kW = child.weight.shape
            r = max(1, int(min(C_out, C_in) * RANK_RATIO))
            print(f"    SVD: replacing Conv({C_in}→{C_out}, {kH}×{kW})  rank={r}", flush=True)
            replacement = SVDConvReplacement(child, rank=r)
            setattr(parent, name, replacement)   # ← use `name`, not `attr_name`

    # Need a wrapper call since named_children is not recursive by itself
    for name, module in list(model.named_children()):
        _replace(model, name, module)

    return model


# ═══════════════════════════════════════════════════════════════════════════════
# DECOMPOSITION 2: Tucker-2 Decomposition
# ═══════════════════════════════════════════════════════════════════════════════
class TuckerConvReplacement(nn.Module):
    """
    Tucker-2 decomposition: compress C_out and C_in modes, keep spatial (kH, kW) intact.

    W ∈ R^(C_out × C_in × kH × kW)
    ≈ G ×₁ A ×₂ B
      G ∈ R^(r_out × r_in × kH × kW)  — core tensor (compressed spatial conv)
      A ∈ R^(C_out × r_out)            — output channel factor matrix
      B ∈ R^(C_in  × r_in)             — input channel factor matrix

    Implemented as three sequential convolutions:
      1) Conv2d(C_in, r_in, 1×1)          ← B  (pointwise input projection)
      2) Conv2d(r_in, r_out, kH×kW, ...)  ← G  (spatial conv in compressed space)
      3) Conv2d(r_out, C_out, 1×1)        ← A  (pointwise output reconstruction)

    Computing A, G, B via HOOI (Higher-Order Orthogonal Iteration):
      - Compute SVD of mode-n unfolding (matricization) of W to get factor matrices
      - Core G = W ×₁ Aᵀ ×₂ Bᵀ
    """
    def __init__(self, original_conv: nn.Conv2d, r_in: int, r_out: int):
        super().__init__()
        W    = original_conv.weight.data.float()      # (C_out, C_in, kH, kW)
        bias = original_conv.bias
        C_out, C_in, kH, kW = W.shape
        stride  = original_conv.stride
        padding = original_conv.padding
        groups  = original_conv.groups

        # ── Mode-1 unfolding (C_out mode): shape (C_out, C_in·kH·kW)
        W_mode1 = W.reshape(C_out, -1)
        U1, _, _ = torch.linalg.svd(W_mode1, full_matrices=False)
        A = U1[:, :r_out]             # (C_out, r_out)  — output factor

        # ── Mode-2 unfolding (C_in mode): shape (C_in, C_out·kH·kW)
        # Permute to bring C_in to front: (C_in, C_out, kH, kW) → (C_in, C_out·kH·kW)
        W_mode2 = W.permute(1, 0, 2, 3).reshape(C_in, -1)
        U2, _, _ = torch.linalg.svd(W_mode2, full_matrices=False)
        B = U2[:, :r_in]              # (C_in, r_in)    — input factor

        # ── Core tensor G = W ×₁ Aᵀ ×₂ Bᵀ
        # W ×₁ Aᵀ: contract C_out mode
        #   W:  (C_out, C_in, kH, kW)
        #   Aᵀ: (r_out, C_out)
        #   result: (r_out, C_in, kH, kW)
        G = torch.einsum("oihw,ro->rihw", W, A.T)   # (r_out, C_in, kH, kW)
        # G ×₂ Bᵀ: contract C_in mode
        #   G:  (r_out, C_in, kH, kW)
        #   Bᵀ: (r_in, C_in)
        #   result: (r_out, r_in, kH, kW)
        G = torch.einsum("rihw,si->rshw", G, B.T)   # (r_out, r_in, kH, kW)

        dtype = original_conv.weight.dtype

        # Conv 1: input projection B  — (r_in, C_in, 1, 1)
        self.conv1 = nn.Conv2d(C_in, r_in, kernel_size=1, bias=False)
        self.conv1.weight = nn.Parameter(B.T.reshape(r_in, C_in, 1, 1).to(dtype))

        # Conv 2: core spatial conv G — (r_out, r_in, kH, kW)
        self.conv2 = nn.Conv2d(r_in, r_out, kernel_size=(kH, kW),
                               stride=stride, padding=padding,
                               groups=groups, bias=False)
        self.conv2.weight = nn.Parameter(G.to(dtype))

        # Conv 3: output reconstruction A — (C_out, r_out, 1, 1)
        self.conv3 = nn.Conv2d(r_out, C_out, kernel_size=1,
                               bias=(bias is not None))
        self.conv3.weight = nn.Parameter(A.reshape(C_out, r_out, 1, 1).to(dtype))
        if bias is not None:
            self.conv3.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        return self.conv3(self.conv2(self.conv1(x)))


def apply_tucker_decomposition(model: nn.Module) -> nn.Module:
    model = copy.deepcopy(model)

    def _replace(parent: nn.Module, name: str, child: nn.Module):
        for attr_name, submodule in child.named_children():
            _replace(child, attr_name, submodule)
        if isinstance(child, nn.Conv2d) and should_decompose(child):
            C_out, C_in, kH, kW = child.weight.shape
            r_out = max(1, int(C_out * RANK_RATIO))
            r_in  = max(1, int(C_in  * RANK_RATIO))
            print(f"    Tucker: Conv({C_in}→{C_out}, {kH}×{kW})  r_in={r_in} r_out={r_out}",
                  flush=True)
            replacement = TuckerConvReplacement(child, r_in=r_in, r_out=r_out)
            setattr(parent, name, replacement)

    for name, module in list(model.named_children()):
        _replace(model, name, module)

    return model


# ═══════════════════════════════════════════════════════════════════════════════
# DECOMPOSITION 3: CP Decomposition (ALS)
# ═══════════════════════════════════════════════════════════════════════════════

def cp_als(W: torch.Tensor, rank: int, n_iters: int = 20) -> list:
    C_out, C_in, kH, kW = W.shape
    W_cpu = W.float().cpu()

    # Random initialization
    A = torch.randn(C_out, rank)
    B = torch.randn(C_in,  rank)
    C = torch.randn(kH,    rank)
    D = torch.randn(kW,    rank)

    # Normalize columns
    for F_ in [A, B, C, D]:
        F_.div_(F_.norm(dim=0, keepdim=True).clamp(min=1e-8))

    def khatri_rao(F1, F2):
        m, r = F1.shape
        n    = F2.shape[0]
        return (F1.unsqueeze(1) * F2.unsqueeze(0)).reshape(m * n, r)

    def hadamard_product(*factors):
        result = factors[0].T @ factors[0]
        for f in factors[1:]:
            result = result * (f.T @ f)
        return result

    def normalize(F):
        """Normalize columns, replace any NaN/Inf rows with random unit vectors."""
        norms = F.norm(dim=0, keepdim=True).clamp(min=1e-8)
        F = F / norms
        # Replace any NaN or Inf with fresh random unit vector
        bad = ~torch.isfinite(F).all(dim=0)
        if bad.any():
            F[:, bad] = torch.nn.functional.normalize(
                torch.randn(F.shape[0], bad.sum()), dim=0)
        return F

    def safe_solve(W_unfold, KR, gram):
        """Solve W_unfold @ KR @ pinv(gram) with fallback if gram is ill-conditioned."""
        # Add small ridge regularization for stability
        reg = 1e-8 * torch.eye(gram.shape[0])
        gram_reg = gram + reg
        try:
            result = W_unfold @ KR @ torch.linalg.pinv(gram_reg)
        except Exception:
            result = W_unfold @ KR @ torch.linalg.pinv(gram + 1e-4 * torch.eye(gram.shape[0]))
        # Fallback: if result has non-finite values, return random init
        if not torch.isfinite(result).all():
            result = torch.randn_like(result)
        return result

    for it in range(n_iters):
        # Update A
        W0   = W_cpu.reshape(C_out, -1)
        KR   = khatri_rao(khatri_rao(D, C), B)
        gram = hadamard_product(B, C, D)
        A    = safe_solve(W0, KR, gram)
        A    = normalize(A)

        # Update B
        W1   = W_cpu.permute(1, 0, 2, 3).reshape(C_in, -1)
        KR   = khatri_rao(khatri_rao(D, C), A)
        gram = hadamard_product(A, C, D)
        B    = safe_solve(W1, KR, gram)
        B    = normalize(B)

        # Update C
        W2   = W_cpu.permute(2, 0, 1, 3).reshape(kH, -1)
        KR   = khatri_rao(khatri_rao(D, B), A)
        gram = hadamard_product(A, B, D)
        C    = safe_solve(W2, KR, gram)
        C    = normalize(C)

        # Update D
        W3   = W_cpu.permute(3, 0, 1, 2).reshape(kW, -1)
        KR   = khatri_rao(khatri_rao(C, B), A)
        gram = hadamard_product(A, B, C)
        D    = safe_solve(W3, KR, gram)
        D    = normalize(D)

    return [A, B, C, D]


class CPConvReplacement(nn.Module):
    """
    CP decomposition of Conv2d via 4 sequential convolutions
    (Lebedev et al. 2014 scheme):

    W ≈ Σᵢ aᵢ⊗bᵢ⊗cᵢ⊗dᵢ   (CP factors: A, B, C, D)

    Implemented as:
      1) Conv2d(C_in, r, 1×1)          ← B factor  (pointwise input)
      2) Conv2d(r, r, kH×1, ...)       ← C factor  (vertical spatial)
      3) Conv2d(r, r, 1×kW, ...)       ← D factor  (horizontal spatial)
      4) Conv2d(r, C_out, 1×1)         ← A factor  (pointwise output)
    """
    def __init__(self, original_conv: nn.Conv2d, rank: int, als_iters: int = 20):
        super().__init__()
        W    = original_conv.weight.data
        bias = original_conv.bias
        C_out, C_in, kH, kW = W.shape
        stride  = original_conv.stride
        padding = original_conv.padding

        print(f"      CP-ALS: Conv({C_in}→{C_out}, {kH}×{kW}) rank={rank} iters={als_iters}",
              flush=True)

        # Run ALS
        A, B, C_f, D = cp_als(W, rank=rank, n_iters=als_iters)
        # A: (C_out, r), B: (C_in, r), C_f: (kH, r), D: (kW, r)

        dtype = original_conv.weight.dtype

        # Depthwise-style spatial convolutions: each of r filters independently
        # Conv 1: pointwise — B factor (C_in → r)
        self.conv1 = nn.Conv2d(C_in, rank, kernel_size=1, bias=False)
        self.conv1.weight = nn.Parameter(B.T.reshape(rank, C_in, 1, 1).to(dtype))

        # Conv 2: vertical spatial — C factor (kH×1)
        pad_h = padding[0] if isinstance(padding, (tuple, list)) else padding
        self.conv2 = nn.Conv2d(rank, rank, kernel_size=(kH, 1),
                               stride=(stride[0] if isinstance(stride, (tuple, list)) else stride, 1),
                               padding=(pad_h, 0),
                               groups=rank, bias=False)  # depthwise over r channels
        self.conv2.weight = nn.Parameter(C_f.T.reshape(rank, 1, kH, 1).to(dtype))

        # Conv 3: horizontal spatial — D factor (1×kW)
        pad_w = padding[1] if isinstance(padding, (tuple, list)) else padding
        self.conv3 = nn.Conv2d(rank, rank, kernel_size=(1, kW),
                               stride=(1, stride[1] if isinstance(stride, (tuple, list)) else stride),
                               padding=(0, pad_w),
                               groups=rank, bias=False)
        self.conv3.weight = nn.Parameter(D.T.reshape(rank, 1, 1, kW).to(dtype))

        # Conv 4: pointwise — A factor (r → C_out)
        self.conv4 = nn.Conv2d(rank, C_out, kernel_size=1,
                               bias=(bias is not None))
        self.conv4.weight = nn.Parameter(A.reshape(C_out, rank, 1, 1).to(dtype))
        if bias is not None:
            self.conv4.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        return self.conv4(self.conv3(self.conv2(self.conv1(x))))


def apply_cp_decomposition(model: nn.Module) -> nn.Module:
    model = copy.deepcopy(model)

    def _replace(parent: nn.Module, name: str, child: nn.Module):
        for attr_name, submodule in child.named_children():
            _replace(child, attr_name, submodule)
        if isinstance(child, nn.Conv2d) and should_decompose(child):
            C_out, C_in, kH, kW = child.weight.shape
            r = max(1, int(min(C_out, C_in) * RANK_RATIO))
            replacement = CPConvReplacement(child, rank=r, als_iters=CP_ALS_ITERS)
            setattr(parent, name, replacement)

    for name, module in list(model.named_children()):
        _replace(model, name, module)

    return model


# ── Decomposition registry ────────────────────────────────────────────────────
DECOMPOSITIONS = {
    "SVD-CONV" : apply_svd_decomposition,
    "TUCKER-2" : apply_tucker_decomposition,
    "CP-APPROX": apply_cp_decomposition,
}


# ── Optional fine-tuning ──────────────────────────────────────────────────────
def finetune(model: nn.Module, train_loader: DataLoader,
             epochs: int, lr: float) -> list:
    """
    Short fine-tuning pass to recover accuracy lost during decomposition.
    Uses standard CE loss (no KD needed — model already knows the task).
    """
    if epochs == 0:
        return []
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    epoch_accs = []
    print(f"      Fine-tuning for {epochs} epochs @ lr={lr} ...", flush=True)
    for epoch in range(epochs):
        correct = total = 0
        for inputs, targets in train_loader:
            inputs  = inputs.to(DEVICE,  non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                loss = F.cross_entropy(model(inputs), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            preds    = model(inputs.detach()).argmax(1)
            correct += (preds == targets).sum().item()
            total   += targets.size(0)
        scheduler.step()
        acc = correct / total
        epoch_accs.append(round(acc, 6))
        print(f"        FT epoch {epoch+1}/{epochs}  train_acc={acc:.4f}", flush=True)
    return epoch_accs


# ── Main experiment runner ────────────────────────────────────────────────────
def run_decomposition(original_model: nn.Module,
                      train_loader: DataLoader,
                      test_loader: DataLoader,
                      method_name: str,
                      baseline_acc: float,
                      original_size_mb: float) -> dict:
    print(f"\n  ┌─ [{method_name}]", flush=True)
    decompose_fn = DECOMPOSITIONS[method_name]

    # ── Step 1: Apply decomposition (no gradient needed)
    t_decomp_start = time.perf_counter()
    print(f"      Applying {method_name} decomposition ...", flush=True)
    decomposed = decompose_fn(original_model)
    decomposed = decomposed.to(DEVICE)
    sync()
    decomp_time_s = time.perf_counter() - t_decomp_start
    print(f"      Decomposition done in {decomp_time_s:.1f}s", flush=True)

    # ── Step 2: Evaluate immediately after decomposition (before fine-tuning)
    pre_ft_metrics = evaluate(decomposed, test_loader, DEVICE)
    print(f"      Pre-FT accuracy: {pre_ft_metrics['accuracy']:.4f}", flush=True)

    # ── Step 3: Fine-tuning (optional)
    ft_epoch_accs = []
    if FINETUNE_EPOCHS > 0:
        ft_epoch_accs = finetune(decomposed, train_loader,
                                 epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR)

    # ── Step 4: Final evaluation
    metrics  = evaluate(decomposed, test_loader, DEVICE)
    infer_ms = measure_inference_ms(decomposed, DEVICE)
    size_mb  = model_size_mb(decomposed)
    acc_drop = baseline_acc - metrics["accuracy"]

    params_decomposed  = param_count(decomposed)
    params_original    = param_count(original_model)
    param_reduction    = 1.0 - params_decomposed / params_original

    print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
          f"Size={size_mb:.2f}MB  Infer={infer_ms:.1f}ms  "
          f"Params={params_decomposed:,} ({param_reduction*100:.1f}% reduction)",
          flush=True)

    return {
        # identification
        "method"                 : "low_rank_decomposition",
        "variant"                : method_name,
        "original_arch"          : MODEL_ARCH,
        "dataset"                : "CIFAR-10",
        # hyperparameters
        "rank_ratio"             : RANK_RATIO,
        "finetune_epochs"        : FINETUNE_EPOCHS,
        "finetune_lr"            : FINETUNE_LR,
        "cp_als_iters"           : CP_ALS_ITERS if method_name == "CP-APPROX" else "N/A",
        "train_device"           : str(DEVICE),
        # decomposition-specific note
        "decomposition_note"     : {
            "SVD-CONV" : f"W(C_out, C_in·kH·kW) → Uᵣ·Σᵣ·Vᵣᵀ, rank={RANK_RATIO}×min(C_out,C_in), 2 conv layers",
            "TUCKER-2" : f"Tucker on (C_out,C_in) modes: G×₁A×₂B, r_out=C_out×{RANK_RATIO}, r_in=C_in×{RANK_RATIO}, 3 conv layers",
            "CP-APPROX": f"CP-ALS rank={RANK_RATIO}×min(C_out,C_in), 4 depthwise conv layers, {CP_ALS_ITERS} ALS iters",
        }[method_name],
        # timing
        "decomposition_time_s"   : round(decomp_time_s, 2),
        # performance metrics
        "pre_finetune_accuracy"  : round(pre_ft_metrics["accuracy"], 6),
        "accuracy"               : round(metrics["accuracy"],  6),
        "precision"              : round(metrics["precision"], 6),
        "recall"                 : round(metrics["recall"],    6),
        "f1"                     : round(metrics["f1"],        6),
        "accuracy_drop"          : round(acc_drop, 6),
        "finetune_epoch_accs"    : ft_epoch_accs,
        # size and efficiency
        "original_params"        : params_original,
        "compressed_params"      : params_decomposed,
        "param_reduction_pct"    : round(param_reduction * 100, 2),
        "size_disk_mb"           : round(size_mb, 4),
        "original_size_mb"       : round(original_size_mb, 4),
        "compression_ratio"      : round(original_size_mb / size_mb, 4) if size_mb else None,
        "inference_ms"           : round(infer_ms, 4),
        # meta
        "description"            : (
            f"Low-rank decomposition ({method_name}): {MODEL_ARCH}, CIFAR-10, "
            f"rank_ratio={RANK_RATIO}, finetune_epochs={FINETUNE_EPOCHS}, device={DEVICE}"
        ),
        "status": "ok",
    }


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Low-Rank Decomposition (SVD / Tucker-2 / CP) — CIFAR-10")
    print(f"  Model   : {MODEL_ARCH}  from {ORIGINAL_MODEL_PATH}")
    print(f"  Device  : {DEVICE}  |  RankRatio: {RANK_RATIO}")
    print(f"  FineTune: {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}")
    print(f"{'='*65}\n")

    # Load baseline metrics
    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    # Load original model
    original_model = load_model(ORIGINAL_MODEL_PATH, MODEL_ARCH)
    original_size_mb = model_size_mb(original_model)
    original_params  = param_count(original_model)
    print(f"  Original size     : {original_size_mb:.2f} MB", flush=True)
    print(f"  Original params   : {original_params:,}\n", flush=True)

    # Data loaders (needed for fine-tuning)
    train_loader, test_loader = get_loaders()

    results = []
    for method_name in DECOMPOSITIONS:
        try:
            row = run_decomposition(
                original_model, train_loader, test_loader,
                method_name, baseline_acc, original_size_mb
            )
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  └─ FAILED [{method_name}]: {e}", flush=True)
            row = {
                "method": "low_rank_decomposition", "variant": method_name,
                "status": "failed", "error": str(e),
                "accuracy": None, "accuracy_drop": None,
                "size_disk_mb": None,
            }
        results.append(row)

    valid = [r for r in results if r.get("accuracy") is not None]
    best  = min(valid, key=lambda r: r["accuracy_drop"]) if valid else None

    report = {
        "experiment"         : "low_rank_decomposition",
        "original_arch"      : MODEL_ARCH,
        "dataset"            : "CIFAR-10",
        "train_device"       : str(DEVICE),
        "rank_ratio"         : RANK_RATIO,
        "finetune_epochs"    : FINETUNE_EPOCHS,
        "finetune_lr"        : FINETUNE_LR,
        "cp_als_iters"       : CP_ALS_ITERS,
        "variants_tested"    : list(DECOMPOSITIONS.keys()),
        "baseline"           : baseline,
        "original_size_mb"   : round(original_size_mb, 4),
        "original_params"    : original_params,
        "best_variant"       : best["variant"] if best else None,
        "best_config": {
            "variant"             : best["variant"]          if best else None,
            "accuracy"            : best["accuracy"]         if best else None,
            "accuracy_drop"       : best["accuracy_drop"]    if best else None,
            "size_disk_mb"        : best["size_disk_mb"]     if best else None,
            "inference_ms"        : best["inference_ms"]     if best else None,
            "compression_ratio"   : best["compression_ratio"] if best else None,
            "param_reduction_pct" : best["param_reduction_pct"] if best else None,
        } if best else {},
        "results": results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if best:
        print(f"  Best variant       : {best['variant']}")
        print(f"  Accuracy           : {best['accuracy']:.4f}  (drop={best['accuracy_drop']:+.4f})")
        print(f"  Pre-FT accuracy    : {best['pre_finetune_accuracy']:.4f}")
        print(f"  Compressed size    : {best['size_disk_mb']:.2f} MB  "
              f"(original={original_size_mb:.2f} MB, "
              f"ratio={best['compression_ratio']:.2f}×)")
        print(f"  Param reduction    : {best['param_reduction_pct']:.1f}%")
        print(f"  Inference          : {best['inference_ms']:.1f} ms")


if __name__ == "__main__":
    main()

[init] PyTorch  : 2.12.0.dev20260324+cu128
[init] Device   : cuda
[init] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[init] Compute  : SM120
[init] Model    : resnet50  from ../__2__baseline_resnet50_cifar10.pth
[init] RankRatio: 0.25  FineTuneEpochs: 5

  Low-Rank Decomposition (SVD / Tucker-2 / CP) — CIFAR-10
  Model   : resnet50  from ../__2__baseline_resnet50_cifar10.pth
  Device  : cuda  |  RankRatio: 0.25
  FineTune: 5 epochs @ lr=0.001

  Baseline accuracy : 0.9320

[model] Loading resnet50 from ../__2__baseline_resnet50_cifar10.pth ...
[model] Loaded. Params: 23,520,842
  Original size     : 90.03 MB
  Original params   : 23,520,842

[data] Train batches: 391  Test batches: 20

  ┌─ [SVD-CONV]
      Applying SVD-CONV decomposition ...
    SVD: replacing Conv(64→64, 3×3)  rank=16
    SVD: replacing Conv(64→64, 3×3)  rank=16
    SVD: replacing Conv(64→64, 3×3)  rank=16
    SVD: replacing Conv(128→128, 3×3)  rank=32
    SVD: replacing Conv(128→128, 3×3)  rank=32
    SVD: replaci